# SSDC 2026 — Data Cleaning: Student Placement System Dataset by kata edward

In [150]:
import pandas as pd
from pathlib import Path

In [151]:
RAW_DIR = Path('data/raw')
CLEAN_DIR  = Path('data/clean')

FILES = {
    "company": ",",
    "talent_request": ",",
    "student_all": ",",
    "tracking_student": ",",
    "tracking_company": ";",
    "status_student": ";",
}

STR_COLS = {"NIM", "pic_phone", "hp", "no_whatsapp"}
DATE_COLS = {"created_at", "request_date", "send_date", "sync_date", "last_update"}

## Ubah separator semua table jadi koma & kolom STR_COLS jadi string

In [152]:
def load_raw(name, sep):
    path = RAW_DIR / f"{name}.csv"
    header = pd.read_csv(path, sep=sep, nrows=0, encoding="utf-8-sig").columns
    dtype  = {c: str for c in header if c.strip() in STR_COLS} # Ubah jadi teks untuk menjaga angka 0 di depan
    df = pd.read_csv(path, sep=sep, dtype=dtype, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    return df

raw = {name: load_raw(name, sep) for name, sep in FILES.items()}

# Semua separator output diseragamkan menggunakan koma
CLEAN_DIR = Path("data/clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

## Ubah format tiap kolom sesuai dengan dokumentasi data set

In [153]:
ISO = "%Y-%m-%d"     # company, talent_request, tracking_student
DMY = "%d/%m/%Y"     # tracking_company, status_student  (dd/mm/yyyy!)

# company
raw["company"]["created_at"] = pd.to_datetime(raw["company"]["created_at"], format=ISO, errors="coerce")

# talent_request
raw["talent_request"]["request_date"] = pd.to_datetime(raw["talent_request"]["request_date"], format=ISO, errors="coerce")

# status_student
ss = raw["status_student"]
ss["no_whatsapp"] = ss["no_whatsapp"].str.replace(r"^(?!0)", "0", regex=True) # nol depan hilang di sumber -> tambahkan
ss["sync_date"]   = pd.to_datetime(ss["sync_date"], format=DMY, errors="coerce") # dd/mm/yyyy
ss["IPK"]         = pd.to_numeric(ss["IPK"], errors="coerce") # DECIMAL -> float

# tracking_company
ts = raw["tracking_student"]
rebuilt = (ts.groupby("id_tracking_company")["NIM"].apply(lambda x: ",".join(sorted(x))).rename("list_nim"))
tc = raw["tracking_company"].drop(columns=["list_nim"]).merge(rebuilt, on="id_tracking_company", how="left")  # list_nim dibangun ulang
tc["request_date"] = pd.to_datetime(tc["request_date"], format=DMY, errors="coerce")  # dd/mm/yyyy
tc["send_date"]    = pd.to_datetime(tc["send_date"],    format=DMY, errors="coerce")  # dd/mm/yyyy
raw["tracking_company"] = tc

# tracking_student
raw["tracking_student"]["last_update"] = pd.to_datetime(raw["tracking_student"]["last_update"], format=ISO, errors="coerce")

## Validasi dengan data raw

In [154]:
print("== 1. Jumlah baris harus identik dengan aslinya ==")
expected = {"company":1500,"talent_request":12000,"student_all":25000,
            "tracking_student":41600,"tracking_company":12000,"status_student":25000}
for name, n in expected.items():
    got = len(raw[name]); print(f"  {name:18s} {got:>6}  {'OK' if got==n else '<-- BEDA!'}")

print("\n== 2. Tanggal gagal parse (harus 0; kecuali send_date=598 Draft) ==")
for name, col in [("company","created_at"),("talent_request","request_date"),
                  ("tracking_student","last_update"),("tracking_company","request_date"),
                  ("tracking_company","send_date"),("status_student","sync_date")]:
    print(f"  {name}.{col:14s}: {raw[name][col].isna().sum()} null")

print("\n== 3. IPK gagal jadi angka (harus 0) ==")
print(f"  status_student.IPK: {raw['status_student']['IPK'].isna().sum()} null")

print("\n== 4. list_nim: jumlah NIM == jumlah_dikirimkan? ==")
tc = raw["tracking_company"]
n_nim = tc["list_nim"].fillna("").apply(lambda s: len([t for t in s.split(",") if t]))
print(f"  proporsi cocok: {(n_nim == pd.to_numeric(tc['jumlah_dikirimkan'])).mean():.1%}")

== 1. Jumlah baris harus identik dengan aslinya ==
  company              1500  OK
  talent_request      12000  OK
  student_all         25000  OK
  tracking_student    41600  OK
  tracking_company    12000  OK
  status_student      25000  OK

== 2. Tanggal gagal parse (harus 0; kecuali send_date=598 Draft) ==
  company.created_at    : 0 null
  talent_request.request_date  : 0 null
  tracking_student.last_update   : 0 null
  tracking_company.request_date  : 0 null
  tracking_company.send_date     : 598 null
  status_student.sync_date     : 0 null

== 3. IPK gagal jadi angka (harus 0) ==
  status_student.IPK: 0 null

== 4. list_nim: jumlah NIM == jumlah_dikirimkan? ==
  proporsi cocok: 100.0%


## Export ke /data/clean

In [155]:
for name, df in raw.items():
    df.to_csv(CLEAN_DIR / f"{name}.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d")
    print("tersimpan:", f"{name}.csv")
print("\nSelesai. File bersih ada di:", CLEAN_DIR)

tersimpan: company.csv
tersimpan: talent_request.csv
tersimpan: student_all.csv
tersimpan: tracking_student.csv
tersimpan: tracking_company.csv
tersimpan: status_student.csv

Selesai. File bersih ada di: data/clean


In [156]:
for table, sep in FILES.items():
    header = pd.read_csv(CLEAN_DIR / f"{table}.csv", nrows=0, encoding="utf-8-sig").columns
    dtype     = {c: str for c in header if c.strip() in STR_COLS}
    date_cols = [c for c in header if c.strip() in DATE_COLS]   # hanya kolom yang ADA di tabel ini
    df = pd.read_csv(CLEAN_DIR / f"{table}.csv", dtype=dtype,
                     parse_dates=date_cols, encoding="utf-8-sig")
    print(f"\n=== {table} ===")
    print(df.info())


=== company ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id_company        1500 non-null   object        
 1   company_name      1500 non-null   object        
 2   company_type      1500 non-null   object        
 3   industry_sector   1500 non-null   object        
 4   kota              1500 non-null   object        
 5   skala_perusahaan  1500 non-null   object        
 6   pic_name          1500 non-null   object        
 7   pic_phone         1500 non-null   object        
 8   created_at        1500 non-null   datetime64[ns]
dtypes: datetime64[ns](1), object(8)
memory usage: 105.6+ KB
None

=== talent_request ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------         